# DocuMind AI
### An AI-Powered Research Paper Assistant Using Retrieval-Augmented Generation (RAG)

**Project Overview**

DocuMind AI is an educational research-paper assistant that uses Retrieval-Augmented Generation (RAG) to answer questions and generate summaries from uploaded research papers.

## Project Workflow

1. PDF text extraction with page numbers
2. Page-aware text chunking
3. BGE embeddings and FAISS vector search
4. BM25 keyword search
5. Hybrid retrieval using Reciprocal Rank Fusion (RRF)
6. Qwen-based question answering with retrieved page references
7. Map-Reduce document summarization
8. Retrieval evaluation

## Technology Stack

- **PDF Processing:** PyMuPDF
- **Tokenization:** BERT Tokenizer
- **Embeddings:** BAAI/bge-base-en-v1.5
- **Vector Database:** FAISS
- **Keyword Retrieval:** BM25
- **Hybrid Retrieval:** Reciprocal Rank Fusion (RRF)
- **Language Model:** Qwen2.5-1.5B-Instruct
- **Deep Learning Framework:** PyTorch
- **Application Interface:** Streamlit

## Project Structure

The development and experimentation workflow is documented in this notebook. The Streamlit application is maintained separately in `DocuMind_Streamlit.py`.

## Note

This is an educational prototype. Generated answers may contain inaccuracies. Retrieved page references identify relevant passages but are not claim-by-claim verified citations.

## 1. Install dependencies

In [ ]:
!pip install -q pymupdf transformers sentence-transformers faiss-cpu rank-bm25 accelerate

## 2. Import libraries and select device

In [ ]:
import os
import numpy as np
import torch
import pymupdf
import faiss

from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google.colab import files

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## 3. Upload and extract the PDF

In [ ]:
uploaded = files.upload()
pdf_path = next(iter(uploaded))

doc = pymupdf.open(pdf_path)
print("PDF:", pdf_path)
print("Pages:", len(doc))

page_data = []
for page_number, page in enumerate(doc, start=1):
    page_data.append({
        "page": page_number,
        "text": page.get_text()
    })

print("Extracted pages:", len(page_data))
print("Total extracted characters:", sum(len(p["text"]) for p in page_data))

if not any(p["text"].strip() for p in page_data):
    raise ValueError("No selectable text was extracted. This PDF may be scanned and require OCR.")

Saving 1706.03762v7.pdf to 1706.03762v7.pdf
PDF: 1706.03762v7.pdf
Pages: 15
Extracted pages: 15
Total extracted characters: 39498


## 4. Create page-aware chunks

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def create_page_chunks(page_data, chunk_size=500, overlap=100):
    page_chunks = []

    for page in page_data:
        tokens = tokenizer(
            page["text"],
            add_special_tokens=False
        )["input_ids"]

        start = 0
        while start < len(tokens):
            end = start + chunk_size
            chunk_text = tokenizer.decode(tokens[start:end])

            page_chunks.append({
                "page": page["page"],
                "text": chunk_text
            })

            start += chunk_size - overlap

    return page_chunks

page_chunks = create_page_chunks(page_data)
page_chunk_texts = [chunk["text"] for chunk in page_chunks]

print("Total chunks:", len(page_chunks))
print("First chunk page:", page_chunks[0]["page"] if page_chunks else "No chunks")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (627 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 33
First chunk page: 1


## 5. Generate BGE embeddings

In [ ]:
embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=device)

bge_embeddings = embedding_model.encode(
    page_chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

bge_vectors = np.asarray(bge_embeddings, dtype="float32")
faiss.normalize_L2(bge_vectors)

print("Embedding shape:", bge_vectors.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embedding shape: (33, 768)


## 6. Build the FAISS vector index

In [ ]:
embedding_dimension = bge_vectors.shape[1]

bge_index = faiss.IndexFlatIP(embedding_dimension)
bge_index.add(bge_vectors)

print("Indexed vectors:", bge_index.ntotal)
print("Embedding dimensions:", embedding_dimension)

Indexed vectors: 33
Embedding dimensions: 768


## 7. Build the BM25 keyword index

In [ ]:
tokenized_corpus = [
    chunk["text"].lower().split()
    for chunk in page_chunks
]

bm25_index = BM25Okapi(tokenized_corpus)

print("BM25 indexed chunks:", len(tokenized_corpus))

BM25 indexed chunks: 33


## 8. Implement hybrid retrieval with RRF

In [ ]:
def hybrid_retrieval(question, top_k=6, candidate_k=10, rrf_k=60):
    # Semantic retrieval using BGE + FAISS
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(query_embedding)

    semantic_k = min(candidate_k, bge_index.ntotal)
    _, bge_indices = bge_index.search(query_embedding, k=semantic_k)

    # Keyword retrieval using BM25
    bm25_scores = bm25_index.get_scores(question.lower().split())
    keyword_indices = bm25_scores.argsort()[::-1][:candidate_k]

    # Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, idx in enumerate(bge_indices[0], start=1):
        if idx >= 0:
            rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0) + 1 / (rrf_k + rank)

    for rank, idx in enumerate(keyword_indices, start=1):
        rrf_scores[int(idx)] = rrf_scores.get(int(idx), 0) + 1 / (rrf_k + rank)

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "page": page_chunks[idx]["page"],
            "text": page_chunks[idx]["text"],
            "rrf_score": float(rrf_scores[idx])
        })

    return results

# Quick retrieval check
sample_results = hybrid_retrieval(
    "What is the Transformer architecture?",
    top_k=3
)

for result in sample_results:
    print(f"Page {result['page']} | RRF score: {result['rrf_score']:.4f}")
    print(result["text"][:400])
    print("-" * 60)

Page 3 | RRF score: 0.0323
figure 1 : the transformer - model architecture. the transformer follows this overall architecture using stacked self - attention and point - wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of figure 1, respectively. 3. 1 encoder and decoder stacks encoder : the encoder is composed of a stack of n = 6 identical layers. each layer has two sub - laye
------------------------------------------------------------
Page 8 | RRF score: 0.0305
table 2 : the transformer achieves better bleu scores than previous state - of - the - art models on the english - to - german and english - to - french newstest2014 tests at a fraction of the training cost. model bleu training cost ( flops ) en - de en - fr en - de en - fr bytenet [ 18 ] 23. 75 deep - att + posunk [ 39 ] 39. 2 1. 0 · 1020 gnmt + rl [ 38 ] 24. 6 39. 92 2. 3 · 1019 1. 4 · 1020 conv
------------------------------------------------------------
Page 8 | RRF score: 0.

## 9. Load the Qwen instruction model

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

new_tokenizer = AutoTokenizer.from_pretrained(model_name)

model_dtype = torch.float16 if device == "cuda" else torch.float32

new_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype
).to(device)

new_model.eval()
print("Qwen model loaded on:", device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen model loaded on: cuda


## 10. Question answering with retrieved context

In [ ]:
def format_answer_with_sources(answer, results):
    pages = sorted({result["page"] for result in results})
    source_text = ", ".join(f"Page {page}" for page in pages)
    return f"{answer}\n\nRetrieved source pages: {source_text}"

def ask_documind_with_citations(question, top_k=6):
    results = hybrid_retrieval(question, top_k=top_k)

    if not results:
        return "I could not retrieve relevant passages from this document."

    context = "\n\n".join(
        f"Page {item['page']}:\n{item['text']}"
        for item in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are DocuMind AI, a research-paper assistant. "
                "Answer only using the supplied document context. "
                "Do not use outside knowledge or invent facts, numbers, "
                "equations, or results. If the context does not contain "
                "the answer, say that the information was not found in "
                "the retrieved passages. Explain clearly and preserve "
                "technical details."
            )
        },
        {
            "role": "user",
            "content": f"Document context:\n{context}\n\nQuestion: {question}"
        }
    ]

    prompt = new_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = new_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(device)

    with torch.inference_mode():
        outputs = new_model.generate(
            **inputs,
            max_new_tokens=350,
            do_sample=False
        )

    answer = new_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return format_answer_with_sources(answer, results)

### Try question answering

In [ ]:
question = "How does scaled dot-product attention work?"
print(ask_documind_with_citations(question))

Scaled dot-product attention involves computing the dot product of the query vector with all keys, dividing each result by the square root of the dimension of the keys, and then applying a softmax function to obtain weights for the values. This process is repeated for multiple heads, allowing the model to attend to different representations of the input data. The formula for scaled dot-product attention is given in equation (1) on page 4.

Retrieved source pages: Page 1, Page 4, Page 5


## 11. Map-Reduce document summarization

In [ ]:
def summarize_chunk(chunk_text):
    messages = [
        {
            "role": "system",
            "content": (
                "Summarize only the technical information present in this "
                "document passage. Preserve important methods, terminology, "
                "and results. Do not invent information. Be concise."
            )
        },
        {
            "role": "user",
            "content": f"Summarize this passage:\n\n{chunk_text}"
        }
    ]

    prompt = new_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = new_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2500
    ).to(device)

    with torch.inference_mode():
        outputs = new_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False
        )

    return new_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )


def summarize_document(page_chunks):
    # MAP: summarize each chunk independently
    chunk_summaries = []

    for i, chunk in enumerate(page_chunks, start=1):
        print(f"Summarizing chunk {i}/{len(page_chunks)}")
        summary = summarize_chunk(chunk["text"])
        chunk_summaries.append(
            f"Page {chunk['page']}: {summary}"
        )

    # REDUCE: combine chunk summaries into one document summary
    combined_summaries = "\n\n".join(chunk_summaries)

    messages = [
        {
            "role": "system",
            "content": (
                "Create a coherent summary of the research paper using "
                "only the supplied chunk summaries. Organize it into "
                "Research Objective, Proposed Method, Architecture, "
                "Experimental Results, and Conclusion. Remove repetition. "
                "Do not invent facts or results."
            )
        },
        {
            "role": "user",
            "content": f"Summarize these section summaries:\n\n{combined_summaries}"
        }
    ]

    prompt = new_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = new_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to(device)

    with torch.inference_mode():
        outputs = new_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    final_summary = new_tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return final_summary


# Run when you want to generate a fresh summary.
# This can take several minutes because each chunk is summarized separately.

### Generate and save the document summary

In [ ]:
final_summary = summarize_document(page_chunks)
print(final_summary)

with open("DocuMind_AI_Final_Summary.txt", "w", encoding="utf-8") as file:
    file.write(final_summary)

files.download("DocuMind_AI_Final_Summary.txt")

Summarizing chunk 1/33
Summarizing chunk 2/33
Summarizing chunk 3/33
Summarizing chunk 4/33
Summarizing chunk 5/33
Summarizing chunk 6/33
Summarizing chunk 7/33
Summarizing chunk 8/33
Summarizing chunk 9/33
Summarizing chunk 10/33
Summarizing chunk 11/33
Summarizing chunk 12/33
Summarizing chunk 13/33
Summarizing chunk 14/33
Summarizing chunk 15/33
Summarizing chunk 16/33
Summarizing chunk 17/33
Summarizing chunk 18/33
Summarizing chunk 19/33
Summarizing chunk 20/33
Summarizing chunk 21/33
Summarizing chunk 22/33
Summarizing chunk 23/33
Summarizing chunk 24/33
Summarizing chunk 25/33
Summarizing chunk 26/33
Summarizing chunk 27/33
Summarizing chunk 28/33
Summarizing chunk 29/33
Summarizing chunk 30/33
Summarizing chunk 31/33
Summarizing chunk 32/33
Summarizing chunk 33/33
### Research Objective:
The primary objective of the research is to evaluate the effectiveness of the Transformer model in achieving state-of-the-art performance on WMT 2014 English-to-German and English-to-French tra

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Small retrieval evaluation

In [ ]:
evaluation_data = [
    {"question": "How does positional encoding work?", "expected_page": 6},
    {"question": "How does multi-head attention work?", "expected_page": 4},
    {"question": "How does the Transformer encoder work?", "expected_page": 3},
    {"question": "What are the experimental results of Transformer?", "expected_page": 8},
    {"question": "How does masked self-attention work?", "expected_page": 5}
]

def evaluate_retrieval(top_k=3):
    correct = 0

    for item in evaluation_data:
        results = hybrid_retrieval(item["question"], top_k=top_k)
        retrieved_pages = [result["page"] for result in results]
        hit = item["expected_page"] in retrieved_pages

        correct += int(hit)
        print(
            f"Question: {item['question']}\n"
            f"Expected page: {item['expected_page']} | "
            f"Retrieved pages: {retrieved_pages} | Hit: {hit}\n"
        )

    recall = correct / len(evaluation_data)
    print(f"Hybrid Recall@{top_k}: {recall:.2%}")
    print("Note: This is a small educational evaluation set, not a broad benchmark.")

evaluate_retrieval(top_k=3)

Question: How does positional encoding work?
Expected page: 6 | Retrieved pages: [6, 6, 9] | Hit: True

Question: How does multi-head attention work?
Expected page: 4 | Retrieved pages: [5, 4, 4] | Hit: True

Question: How does the Transformer encoder work?
Expected page: 3 | Retrieved pages: [3, 8, 2] | Hit: True

Question: What are the experimental results of Transformer?
Expected page: 8 | Retrieved pages: [9, 8, 10] | Hit: True

Question: How does masked self-attention work?
Expected page: 5 | Retrieved pages: [5, 7, 5] | Hit: True

Hybrid Recall@3: 100.00%
Note: This is a small educational evaluation set, not a broad benchmark.


## 13. Project limitations

- The application works best with PDFs that contain selectable text. Scanned PDFs need OCR.
- Retrieval quality depends on the document, chunking, embedding model, and query.
- Qwen can still generate unsupported or inaccurate statements despite grounding instructions.
- Displayed page references identify retrieved passages; they do not prove that every generated claim is supported by those pages.
- The retrieval evaluation uses only five questions and should not be presented as a robust benchmark.
- The notebook uses Colab GPU when available. A fresh Colab runtime requires rerunning setup, uploading the PDF, and rebuilding the indexes.

## Final implementation summary

**Final retrieval path:** PDF → page-aware chunks → BGE embeddings + FAISS, alongside BM25 → RRF hybrid retrieval → Qwen answer generation → retrieved page references.

**Final summarization path:** page-aware chunks → chunk summaries (Map) → combined summary (Reduce).

The Streamlit user interface is maintained separately in `app_docuMind_AI.py`.